# DenseNet–ViT GPU 비교
하이퍼파라미터는 저장소 최상위의 `params.yaml`에서 바꿉니다. 이 노트북은 최신 저장소 코드와 설정을 읽어 실행합니다. 조건을 바꾸면 `experiment_id`도 새 이름으로 지정하세요.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'params.yaml').is_file()), None)
assert ROOT is not None, '저장소 폴더에서 Jupyter를 시작하세요.'
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
from src.experiment_config import load_config, validate_training
CONFIG = ROOT / 'params.yaml'  # 다른 설정 파일은 이 경로만 변경
config = validate_training(load_config(CONFIG))
DATA_ARCHIVE = ROOT / 'vit_clahe_colab.zip'
print('Python:', sys.executable)
print(json.dumps({k:config[k] for k in ('experiment_id','models','seeds','epochs','batch_size','lr','optimizer','weight_decay','pos_weight','clahe','vit_last_block')}, indent=2))

In [ ]:
if config['device'] == 'cuda':
    from scripts.check_gpu import check_gpu
    print(json.dumps(check_gpu(), indent=2))
else:
    print('CPU mode selected in params.yaml')

In [ ]:
from scripts.prepare_gpu_data import prepare
prepare(CONFIG, DATA_ARCHIVE if DATA_ARCHIVE.is_file() else None);

## 소량 실행 점검
학습 영상 32장만 사용하는 점검입니다. GPU 컴퓨터에서 첫 실행 또는 학습 코드 변경 시 진행하세요.

In [ ]:
from scripts.run_comparison import main as run_comparison
source_hash = hashlib.sha256(b''.join(p.read_bytes() for p in sorted((ROOT / 'src').glob('*.py')))).hexdigest()[:12]
run_comparison(['--config', str(CONFIG), '--smoke',
                '--output-root', str(ROOT / config['output_root'] / 'execution_checks' / source_hash)])

## 본 학습
DenseNet 다음 ViT를 학습합니다. 중단되면 같은 셀을 다시 실행해 마지막 체크포인트부터 이어갑니다.

In [ ]:
run_comparison(['--config', str(CONFIG)])

In [ ]:
import pandas as pd
config = validate_training(load_config(CONFIG))
output = ROOT / config['output_root'] / config['experiment_id']
display(pd.read_csv(output / 'comparison.csv'))
print('결과와 params.resolved.yaml을 함께 보관하세요:', output)